In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_file = Path(
    r"C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data\observations_clean.parquet"
)
df = pd.read_parquet(data_file)

df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
df["pm25"] = pd.to_numeric(df["pm25"], errors="coerce")
df = (
    df.dropna(subset=["station", "event_time", "pm25"])
    .sort_values(["station", "event_time"])
    .copy()
)

# Variables calendaires cycliques
hour_angle = 2 * np.pi * df["event_time"].dt.hour / 24
month_angle = 2 * np.pi * (df["event_time"].dt.month - 1) / 12
df["hour_sin"] = np.sin(hour_angle)
df["hour_cos"] = np.cos(hour_angle)
df["month_sin"] = np.sin(month_angle)
df["month_cos"] = np.cos(month_angle)
df["day_of_week"] = df["event_time"].dt.dayofweek

# Retards PM2.5 seulement si l’observation correspond à l’heure exacte
for lag in (1, 2, 3):
    previous_time = df.groupby("station")["event_time"].shift(lag)
    previous_pm25 = df.groupby("station")["pm25"].shift(lag)
    expected_time = df["event_time"] - pd.Timedelta(hours=lag)
    df[f"pm25_lag{lag}"] = previous_pm25.where(previous_time == expected_time)

# Cible : PM2.5 une heure plus tard, uniquement si l’heure suivante existe
next_time = df.groupby("station")["event_time"].shift(-1)
next_pm25 = df.groupby("station")["pm25"].shift(-1)
df["pm25_target_next_hour"] = next_pm25.where(
    next_time == df["event_time"] + pd.Timedelta(hours=1)
)

required = [
    "pm25_lag1", "pm25_lag2", "pm25_lag3",
    "pm25_target_next_hour",
]
features = df.dropna(subset=required).copy()

output_file = Path(
    r"C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data\pm25_features.parquet"
)
output_file.parent.mkdir(parents=True, exist_ok=True)
features.to_parquet(output_file, index=False)

print(f"Lignes avec variables complètes : {len(features):,}")
print(f"Fichier sauvegardé : {output_file}")
display(features.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9104\3967617130.py:31: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  expected_time = df["event_time"] - pd.Timedelta(hours=lag)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9104\3967617130.py:31: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  expected_time = df["event_time"] - pd.Timedelta(hours=lag)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9104\3967617130.py:31: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  expected_time = df["event_time"] 

Lignes avec variables complètes : 401,706
Fichier sauvegardé : C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data\pm25_features.parquet


,No,year,month,day,hour,pm25,PM10,SO2,NO2,CO,...,event_date,hour_sin,hour_cos,month_sin,month_cos,day_of_week,pm25_lag1,pm25_lag2,pm25_lag3,pm25_target_next_hour
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,...,2013-03-01,0.707107,7.071068e-01,0.866025,0.5,4,7.0,8.0,4.0,3.0
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,...,2013-03-01,0.866025,5.000000e-01,0.866025,0.5,4,6.0,7.0,8.0,5.0
5,6,2013,3,1,5,5.0,5.0,18.0,18.0,400.0,...,2013-03-01,0.965926,2.588190e-01,0.866025,0.5,4,3.0,6.0,7.0,3.0
6,7,2013,3,1,6,3.0,3.0,18.0,32.0,500.0,...,2013-03-01,1.000000,6.123234e-17,0.866025,0.5,4,5.0,3.0,6.0,3.0
7,8,2013,3,1,7,3.0,6.0,19.0,41.0,500.0,...,2013-03-01,0.965926,-2.588190e-01,0.866025,0.5,4,3.0,5.0,3.0,3.0
